# 실제 파이프라인 전체 실행·단계별 점검

기준: [개발 계획](../docs/DEV_PLAN.md), [협업 계약](../docs/CONTRACTS.md), [환경 세팅](../docs/SETUP.md).

**전체 실행:** 커널을 저장소의 `.venv/bin/python` (Python 3.11)로 선택하고 재시작한 뒤, 기본 `EXECUTION_MODE = "full"` 상태에서 **Run All / 모두 실행**을 누릅니다. 모든 후보를 실제 그래프 순서로 처리하고 실행 로그·단계별 결과·오류·보고서를 확인합니다.

**한 단계씩:** `EXECUTION_MODE = "step"`으로 바꾸고 준비 셀과 아래 단계별 실행 셀을 하나씩 실행합니다. 전체 실행 셀은 자동으로 건너뜁니다.

실제 흐름은 `load → select → eligibility → technology → market → competitor → investment → save → next_index`이며, 제외·보류·오류·재검색 분기는 기존 `graph.py`가 결정합니다. 모든 후보 처리 후 `rank → report`로 이어집니다. 같은 실제 노드와 입력을 사용하고, 각 노드 뒤 체크포인트에서 결과를 기록합니다. 전체 모드는 이 체크포인트를 자동으로 이어 실행합니다.

C 적격성은 저장된 G4·외부 검증 결과를 읽고, D는 실제 임베딩 검색과 OpenAI 분석, C 경쟁사는 RAG·Tavily·OpenAI, E는 실제 판단·보고서 코드를 사용합니다. 모델·검색 캐시가 없으면 처음에 다운로드와 색인이 필요할 수 있습니다.

**전체 실행 완료와 모든 분석 노드 검증 완료는 다릅니다.** 자격 요건이 확인불가이거나 부적격이면 D·경쟁사·투자 분석으로 가지 않습니다. 아래 실행 범위 표에서 실제 호출된 노드와 호출되지 않은 노드를 구분합니다. 임의 적격 판정이나 가짜 분석을 넣지 않았습니다. 현재 저장된 C01의 G1은 확인불가이므로 `확인필요 → hold`가 예상됩니다.

외부 요청 오류는 실제 그래프의 오류 처리대로 기록됩니다. 후보 적재·최종 보고서처럼 그래프에서 예외를 감싸지 않은 단계의 오류는 셀의 실제 예외로 표시됩니다. API 키는 출력하거나 로그에 저장하지 않습니다.


## 준비 1. 프로젝트·환경 불러오기

기존 `.env`를 불러옵니다. API 키의 내용은 출력하지 않습니다. 여기서는 에이전트를 실행하지 않습니다.

In [ ]:
import json
from collections import Counter
from time import perf_counter
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "graph.py").is_file() and (p / "docs/CONTRACTS.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("skala-rag 저장소 안에서 노트북을 열어주세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from langgraph.checkpoint.memory import InMemorySaver
from config import DATA_DIR, EVALUATION_CRITERIA, OUTPUT_DIR
from graph import build_graph

print(f"프로젝트: {ROOT}")
print(f"커널: {sys.executable}")


## 실행 방식 선택

기본값 `full`은 전체 후보를 순서대로 실행합니다. `step`은 아래 단계별 셀에서 실제 다음 노드 하나만 실행합니다. `SHOW_EACH_RESULT=False`일 때도 모든 출력은 실행 기록에 보존됩니다. 전체 실행 중에는 간단한 순서 로그를 보고, 뒤에서 원하는 기업·노드의 전체 결과를 펼쳐 확인할 수 있습니다.


In [ ]:
EXECUTION_MODE = "full"  # "full": 전체 실행 / "step": 한 노드씩 실행
PAUSE_AFTER_NODE = None  # 예: "technology"이면 기술 노드 하나가 끝난 뒤 정지
SHOW_EACH_RESULT = False  # 전체 실행 중 각 노드의 상세 JSON도 표시하려면 True
if EXECUTION_MODE not in {"full", "step"}:
    raise ValueError("EXECUTION_MODE는 full 또는 step이어야 합니다.")
print("실행 방식:", EXECUTION_MODE)


## 준비 2. 실제 그래프·실행 기록 만들기

`app.py`와 같은 입력으로 실제 그래프를 구성하고, 노드 뒤 체크포인트를 추가합니다. 전체 모드에서는 자동 재개하고 단계별 모드에서는 셀 입력을 기다립니다. 각 노드의 출력·기업ID·소요시간을 `history`에 기록합니다.

이 셀을 다시 실행하면 현재 실행 상태가 초기화됩니다. 첫 기업은 실제 `select` 노드가 디렉토리북 순서대로 선택합니다.


In [ ]:
graph = build_graph().builder.compile(checkpointer=InMemorySaver(), interrupt_after="*")
RUN_CONFIG = {"configurable": {"thread_id": "notebook-pipeline"}, "recursion_limit": 1000}
INITIAL_INPUT = {
    "source_document": str(DATA_DIR / "raw" / "01_기업정보.pdf"),
    "as_of_date": "2026-09-30",
    "evaluation_criteria": EVALUATION_CRITERIA,
}
started = False
history = []
last_node = None
last_update = None


def current_state():
    return graph.get_state(RUN_CONFIG).values if started else {}


def next_nodes():
    return tuple(graph.get_state(RUN_CONFIG).next) if started else ("load",)


def company_summary(company):
    return {key: company.get(key) for key in ("company_id", "기업명", "메인아이템", "sub_domain", "source_page")}


def show_update(update):
    # 표시만 간추립니다. 실제 그래프의 기업 원문·근거·분석은 그대로입니다.
    view = dict(update)
    if "candidate_companies" in view:
        companies = view.pop("candidate_companies")
        view["적재된 기업 수"] = len(companies)
        view["첫 후보"] = company_summary(companies[0]) if companies else None
    if view.get("current_company"):
        view["current_company"] = company_summary(view["current_company"])
    if "evaluation_results" in view:
        view["evaluation_results"] = [
            {**record, "current_company": company_summary(record["current_company"])}
            for record in view["evaluation_results"]
        ]
    print(json.dumps(view, ensure_ascii=False, indent=2, default=str))


def result_summary(node, update, state):
    if update.get("errors"):
        return "오류: " + "; ".join(error["error"] for error in update["errors"])
    if node == "load":
        return f"후보 {len(update.get('candidate_companies', []))}개 적재"
    if node == "select":
        return (update.get("current_company") or {}).get("기업명", "")
    if node == "eligibility":
        data = update.get("eligibility") or {}
        details = ", ".join(f"{key}={data.get(key, {}).get('결과', '?')}" for key in ("G1", "G2", "G3", "G4"))
        return f"{data.get('판정')} | {details}"
    if node in {"technology", "market"}:
        data = update.get(f"{node}_analysis") or {}
        return f"근거충분={data.get('근거충분')} | 호출 {update.get('retrieve_count')} | 추가 근거 {len(update.get('current_evidence', []))}개"
    if node == "competitor":
        data = update.get("competitor_analysis") or {}
        return f"경쟁제품 {len(data.get('경쟁제품', []))}개 | {data.get('비교한계', '')}"
    if node == "investment":
        return f"{update.get('decision')} | 총점 {(update.get('scorecard') or {}).get('total')}"
    if node in {"hold", "exclude", "handle_error"}:
        return f"{update.get('decision')} | {update.get('route_reason', '')}"
    if node == "save":
        return f"기업 결과 누적 {len(state.get('evaluation_results', []))}개"
    if node == "next_index":
        return f"다음 기업 순번 {update.get('current_index')}"
    if node == "rank":
        return f"투자 적격 {len(update.get('ranking', []))}개 | 선정 {update.get('selected_company_id')}"
    if node == "report":
        return f"보고서 {len(update.get('final_report', ''))}자 생성"
    return "완료"


def run_next(expected=None):
    """실제 다음 노드 하나를 실행하고 출력·기업·시간을 기록합니다."""
    global started, last_node, last_update
    upcoming = next_nodes()
    if not upcoming:
        print("전체 흐름이 완료됐습니다.")
        return
    if expected is not None and expected not in upcoming:
        print(f"현재 다음 노드: {', '.join(upcoming)} / {expected} 단계는 실행하지 않았습니다.")
        return
    company = current_state().get("current_company") or {}
    print(f"[{len(history) + 1:03d}] 실행 중: {', '.join(upcoming)} / {company.get('company_id', '-')}", flush=True)
    payload = None if started else INITIAL_INPUT
    tick = perf_counter()
    updates = []
    for event in graph.stream(payload, RUN_CONFIG, stream_mode="updates"):
        for node, update in event.items():
            if node != "__interrupt__":
                updates.append((node, update))
    started = True
    state = current_state()
    company = state.get("current_company") or {}
    elapsed = round(perf_counter() - tick, 3)
    for node, update in updates:
        last_node, last_update = node, update
        summary = result_summary(node, update, state)
        history.append({
            "단계": len(history) + 1, "노드": node,
            "company_id": company.get("company_id") if node not in {"load", "rank", "report"} else None,
            "소요초": elapsed, "요약": summary, "출력": update,
        })
        print(f"[{len(history):03d}] 완료: {node} | {elapsed:.3f}초 | {summary}", flush=True)
        if EXECUTION_MODE == "step" or SHOW_EACH_RESULT:
            show_update(update)
    if EXECUTION_MODE == "step":
        upcoming = next_nodes()
        print("다음 노드:", ", ".join(upcoming) if upcoming else "없음 — 전체 실행 완료")


print(f"준비 완료. 실행 방식: {EXECUTION_MODE}")


## LLM 요청 진행 로그·긴 대기 진단

다음 셀은 로그·진단 함수를 정의합니다. 에이전트 호출이나 실행 상태 초기화는 하지 않습니다. 전체 실행에는 LLM 시작·완료·도구 호출·출력 형식 오류 피드백 로그를 추가합니다. 프롬프트 전체나 API 키는 출력하지 않습니다.

실행 중 기술·시장 단계에서 오래 기다리다가 중지한 경우, **커널을 재시작하지 않고 이 정의 셀만 실행한 뒤** 새 셀에서 `diagnose_current_node()`를 호출할 수 있습니다. 같은 체크포인트의 실제 기술/시장 노드만 재실행하며, 전체 후보를 다시 실행하지 않습니다.

진단 호출은 요청별 타임아웃 30초·네트워크 재시도 0회·LLM 요청 최대 4회로 제한하고 설정을 끝나면 복원합니다. 이 제한으로 실패하면 해당 진단 오류가 실제 그래프의 `errors`에 기록됩니다. 정상 분석 성공으로 해석하지 마세요. 이 진단은 현재 커널에서 요청이 중지된 뒤 사용하는 동기 실행용이며, 이미 진행 중인 요청의 설정을 바꾸지 않습니다.


In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class PipelineLLMTrace(BaseCallbackHandler):
    raise_error = True

    def __init__(self, max_calls=None):
        self.max_calls = max_calls
        self.calls = 0
        self.begins = {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        if self.max_calls is not None and self.calls >= self.max_calls:
            raise RuntimeError(f"진단 제한: LLM 요청 {self.max_calls}회 초과. 출력 형식·도구 반복 로그 확인 필요")
        self.calls += 1
        self.begins[run_id] = (self.calls, perf_counter())
        print(f"  [LLM {self.calls}] 요청 시작", flush=True)
        for conversation in messages:
            for message in conversation[-3:]:
                content = getattr(message, "content", "")
                if getattr(message, "type", None) == "tool" and isinstance(content, str):
                    if "Error:" in content or "validation error" in content.lower() or "Failed to parse" in content:
                        print("  출력 형식 재시도 피드백:", content[:1000], flush=True)

    def on_llm_end(self, response, *, run_id, **kwargs):
        number, tick = self.begins.pop(run_id, ("?", perf_counter()))
        names = []
        for batch in response.generations:
            for generation in batch:
                message = getattr(generation, "message", None)
                names.extend(tool.get("name", "?") for tool in getattr(message, "tool_calls", []))
        print(f"  [LLM {number}] 응답 완료 {perf_counter() - tick:.2f}초 / 도구: {', '.join(names) or '없음'}", flush=True)

    def on_llm_error(self, error, *, run_id, **kwargs):
        number, tick = self.begins.pop(run_id, ("?", perf_counter()))
        print(f"  [LLM {number}] 오류 {type(error).__name__} / {perf_counter() - tick:.2f}초", flush=True)

    def on_tool_start(self, serialized, input_str, **kwargs):
        print("  검색 도구 시작:", (serialized or {}).get("name", "?"), flush=True)

    def on_tool_end(self, output, **kwargs):
        print("  검색 도구 완료", flush=True)


# 현재 체크포인트를 유지하고 이후 실행의 관찰 로그만 추가합니다.
RUN_CONFIG["callbacks"] = [PipelineLLMTrace()]


def diagnose_current_node(timeout=30, max_model_calls=4):
    import llm

    upcoming = next_nodes()
    if not started or upcoming not in {("technology",), ("market",)}:
        print("진단 대상은 중지 후의 실제 기술·시장 노드입니다. 현재 다음 노드:", upcoming)
        return
    model = llm.get_llm()
    original = (model.root_client, model.client, model.request_timeout, model.max_retries)
    old_callbacks = RUN_CONFIG.get("callbacks", [])
    old_mode = globals()["EXECUTION_MODE"]
    bounded = model.root_client.with_options(timeout=timeout, max_retries=0)
    trace = PipelineLLMTrace(max_calls=max_model_calls)
    try:
        model.root_client = bounded
        model.client = bounded.chat.completions
        model.request_timeout = timeout
        model.max_retries = 0
        RUN_CONFIG["callbacks"] = [trace]
        globals()["EXECUTION_MODE"] = "step"
        print(f"현재 {upcoming[0]}만 진단: 요청 타임아웃 {timeout}초 / 재시도 0 / LLM 최대 {max_model_calls}회", flush=True)
        run_next(upcoming[0])
    finally:
        model.root_client, model.client, model.request_timeout, model.max_retries = original
        RUN_CONFIG["callbacks"] = old_callbacks
        globals()["EXECUTION_MODE"] = old_mode
        print(f"진단 종료. LLM 요청 {trace.calls}회. 기존 요청 설정을 복원했습니다.", flush=True)


print("LLM 진행 로그·현재 기술/시장 단계 진단 준비 완료. 실행 상태는 유지됩니다.")


## 전체 파이프라인 실제 실행

`full` 모드에서 모든 후보를 실제 다음 노드 순서대로 실행합니다. 화면의 번호는 실제 실행 순서이며, 기술·시장 재검색도 별도 호출로 기록됩니다. 각 기업의 적격성 결과에 따라 정상 분석·제외·보류·오류 경로를 그대로 따릅니다.

이 셀을 실행하면 `rank → report`까지 자동 진행합니다. 기존 단계별 실행 셀은 `full` 모드에서 추가 노드를 실행하지 않습니다. 결과를 다시 검증하려면 커널을 재시작하고 위에서부터 실행하세요.

`PAUSE_AFTER_NODE = "technology"`로 설정하면 실제 기술 노드 하나가 끝난 뒤 멈춥니다. 형식 오류를 수정한 뒤 첫 기술 결과만 확인할 때 사용하세요. 계속 진행하려면 `PAUSE_AFTER_NODE = None`으로 바꾸고 전체 실행 셀을 다시 실행합니다. D 내부의 모델 요청은 한 분석 호출당 최대 6회로 제한하며, 한도를 넘으면 실제 오류 경로로 기록됩니다. 이 한도는 graph의 기술·시장 재검색 1회 규칙과 별개입니다.


In [ ]:
if EXECUTION_MODE == "full":
    while next_nodes():
        run_next()
        if PAUSE_AFTER_NODE is not None and last_node == PAUSE_AFTER_NODE:
            print(f"확인용 정지: {last_node} 실행 완료 / 실제 다음 노드: {next_nodes()}")
            break
    if next_nodes():
        print("체크포인트에서 멈췄습니다. 실행 기록을 확인한 뒤 이 셀을 다시 실행하면 이어집니다.")
    else:
        print(f"전체 실행 종료: 실제 노드 호출 {len(history)}회 / 기업 결과 {len(current_state().get('evaluation_results', []))}개")
else:
    print("step 모드: 아래 단계별 실행 셀에서 한 노드씩 진행하세요.")


## 실제 실행 범위

실제로 호출된 노드를 확인합니다. 정상 그래프 분기 때문에 호출되지 않은 분석 노드는 미실행으로 표시합니다. 아래 표는 호출 여부이며, 내용의 정확성이나 분석 성공을 보장하는 표는 아닙니다. 오류는 실행 요약과 기업별 상세 출력에서 확인하세요.


In [ ]:
counts = Counter(item["노드"] for item in history)
rows = ["| 실제 노드 | 호출 횟수 | 상태 |", "|---|---:|---|"]
for node in ("load", "select", "eligibility", "technology", "market", "competitor", "investment", "exclude", "hold", "handle_error", "save", "next_index", "rank", "report"):
    count = counts[node]
    rows.append(f"| {node} | {count} | {'실행됨' if count else '미실행'} |")
display(Markdown("\n".join(rows)))
not_called = [node for node in ("technology", "market", "competitor", "investment") if not counts[node]]
if not_called:
    print("실제 분석이 실행되지 않은 단계:", ", ".join(not_called))
    print("기업별 적격성·보류·제외·오류 기록을 확인하세요. 이 단계들은 이번 실행으로 검증되지 않았습니다.")


## 원하는 기업·노드의 결과를 실행 순서대로 보기

실제 실행 중 기록한 출력을 확인합니다. 이 셀은 에이전트를 다시 호출하지 않습니다. `REVIEW_COMPANY_ID`에 C01~C45를 지정하고, `REVIEW_NODE`에 technology·market 등 노드명을 지정할 수 있습니다. `None`이면 해당 필터를 적용하지 않습니다. 재검색 결과도 실제 호출 순서대로 표시됩니다.


In [ ]:
REVIEW_COMPANY_ID = "C01"
REVIEW_NODE = None
review = [item for item in history
          if (REVIEW_COMPANY_ID is None or item["company_id"] == REVIEW_COMPANY_ID)
          and (REVIEW_NODE is None or item["노드"] == REVIEW_NODE)]
if not review:
    print("해당 기업·노드의 실행 기록이 없습니다.")
for item in review:
    print(f"\n===== {item['단계']:03d}: {item['노드']} / {item['company_id']} / {item['소요초']}초 =====")
    show_update(item["출력"])


# 단계별 실행 셀

다음 1~16단계 실행 셀은 **`step` 모드에서만** 노드를 호출합니다. `full` 모드로 전체 실행한 경우에는 각 단계 설명을 참고하고, 위의 상세 결과 셀에서 실제 출력을 확인하세요. 뒤의 실행 요약·보고서·기록 저장 셀은 두 모드 모두 사용할 수 있습니다.


## 1. 후보 적재 — load

**실제 실행:** `agents.loader.run()`이 기업 레코드를 적재합니다. 기존 추출 캐시가 있으면 재사용하고, 없으면 원본 PDF에서 추출합니다.

**확인:** 적재 결과와 첫 후보의 실제 필드, 다음 노드가 `select`인지 봅니다. 전체 기업 데이터는 `current_state()["candidate_companies"]`로 확인할 수 있습니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("load")


## 2. 현재 기업 선택 — select

**실제 실행:** 그래프가 현재 순번의 기업을 선택하고 이전 기업의 분석·근거·재검색 횟수를 초기화합니다.

**확인:** `current_company`, `retrieve_count=0`, 분석 결과 초기화, 다음 노드 `eligibility`.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("select")


## 3. 적격성 — eligibility

**실제 실행:** 해당 기업의 G4 스크리닝 및 저장된 외부 검증 결과로 G1~G4를 판정합니다. 이 노드 실행 자체가 DART/Tavily를 새로 조회하는 것은 아닙니다.

**확인:** `적격`이면 `technology`, `부적격`이면 `exclude`, `확인필요`이면 `hold`, 실행 오류이면 `handle_error`로 이어집니다. 현재 저장된 결과에서 C01은 G4 통과지만 G1이 확인불가이므로 `확인필요 → hold` 경로가 예상됩니다. 이는 실행 오류와 다릅니다. 실제 출력과 근거를 확인하세요.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("eligibility")


## 4. 적격성 결과에 따른 분기

실제 다음 노드가 `exclude`, `hold`, `handle_error`인 경우에만 그 노드 하나를 실행합니다. `technology`로 진행하는 경우 이 셀은 건너뜁니다.

오류를 먼저 수정하려면 여기서 실행을 멈추세요. 이미 import한 모듈의 코드를 변경했다면 커널을 다시 시작하고 준비 셀부터 재실행해야 새 코드가 반영됩니다.

In [ ]:
if EXECUTION_MODE == "step":
    if any(node in {"exclude", "hold", "handle_error"} for node in next_nodes()):
        run_next()
    else:
        print("적격성 단계의 제외·보류·오류 분기가 없습니다. 다음 노드:", next_nodes())


## 5. 기술 분석 첫 호출 — technology

**실제 실행:** 기업 정보로 한국어·영문 질의를 생성하고 기술 문서를 하이브리드 검색한 뒤, 실제 LLM 분석과 원문 근거 검증을 수행합니다.

**확인:** 핵심기술, 성능지표·측정조건, 업계 기준대조, DIR/TEC 근거ID, 미확인정보, `근거충분`, `retrieve_count`. 모델·인덱스가 없으면 첫 실행에 다운로드와 색인이 필요할 수 있습니다.

근거가 부족하면 다음 노드는 다시 `technology`입니다. 오류면 `handle_error`, 충분하면 `market`입니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("technology")


## 6. 기술 재검색 — 필요한 경우에만

그래프가 실제로 `technology`를 다시 선택한 경우에만 실행합니다. 질의 재작성·근거 추가·횟수 증가를 확인합니다. 재검색이 필요하지 않으면 추가 API 호출 없이 건너뜁니다.

In [ ]:
if EXECUTION_MODE == "step":
    if next_nodes() == ("technology",):
        run_next("technology")
    else:
        print("기술 재검색 대상이 아닙니다. 다음 노드:", next_nodes())


## 7. 시장 분석 첫 호출 — market

**실제 실행:** 시장 문서 검색, LLM 분석, 수치·출처·기준연도·성장 기간 검증을 수행합니다.

**확인:** 목표고객, 시장규모, 성장률, 사업모델, 위험, MKT 근거ID, `근거충분`. 시장 전체 수치를 해당 기업의 목표 시장으로 오인하지 않았는지 봅니다. 기술 단계 근거도 현재 기업 안에서 누적됩니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("market")


## 8. 시장 재검색 — 필요한 경우에만

실제 다음 노드가 `market`일 때 한 번 더 실행합니다. 새 질의·미확인정보 보완·횟수 증가를 확인합니다.

In [ ]:
if EXECUTION_MODE == "step":
    if next_nodes() == ("market",):
        run_next("market")
    else:
        print("시장 재검색 대상이 아닙니다. 다음 노드:", next_nodes())


## 9. 경쟁사 비교 — competitor

**실제 실행:** 공유 RAG와 웹 근거를 이용해 경쟁사 비교를 수행합니다.

**확인:** 경쟁제품, 비교표, 비교조건·한계, 근거ID. `TAVILY_API_KEY`나 `OPENAI_API_KEY`, 선행 분석 또는 검색 근거가 부족하면 빈 비교 결과와 비교한계가 반환될 수 있으므로 내용까지 확인하세요.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("competitor")


## 10. 투자 판단 — investment

**실제 실행:** 체크리스트·항목별 점수·근거를 LLM이 작성하고, 총점과 최종 판정은 공용 코드로 계산합니다.

**확인:** 11문항, 15개 점수 항목, `확인불가` 처리, 총점, `decision`, 판단사유. 앞 단계 오류로 여기까지 도달하지 못했다면 이 셀은 실행하지 않습니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("investment")


## 11. 분석 중 오류 분기 — 필요한 경우에만

기술·시장·경쟁사·투자 판단 도중 오류가 발생했다면 실제 그래프의 `handle_error`가 해당 기업을 보류 처리합니다. 예외를 성공 결과로 대체하지 않습니다.

In [ ]:
if EXECUTION_MODE == "step":
    if next_nodes() == ("handle_error",):
        run_next("handle_error")
    else:
        print("분석 오류 분기가 없습니다. 다음 노드:", next_nodes())


## 12. 기업 결과 저장 — save

**실제 실행:** 현재 기업의 분석·판정·근거를 `evaluation_results`에 누적합니다.

**확인:** 저장된 `company_id`, 판정, 오류 사유, 해당 기업의 근거. 보류·제외도 실제 결과로 저장되는지 확인합니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("save")


## 13. 다음 기업 순번 — next_index

**실제 실행:** 기업 순번을 하나 증가시키고 남은 후보가 있으면 `select`, 없으면 `rank`로 이동합니다.

**확인:** 순번 증가와 실제 다음 노드. 아직 전체 후보를 처리하지 않았다면 순위·보고서 단계를 강제로 실행하지 않습니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("next_index")


## 14. 이후 기업도 실제 다음 노드 하나씩 진행

다음 기업은 2~13단계 셀을 다시 실행해도 되고, 아래 셀을 반복 실행해도 됩니다. **아래 셀도 한 번에 노드 하나만 실행합니다.** 자동으로 전체 기업을 처리하는 루프는 넣지 않았습니다.

각 실행 결과를 확인한 뒤 다음 실행으로 넘어가세요. 반복 선택 시 앞 기업의 근거·분석이 초기화되는지 확인하고, 오류가 반복되면 원인을 수정한 후 다시 시작하세요.

In [ ]:
if EXECUTION_MODE == "step":
    run_next()


## 15. 최종 순위 — rank

**실제 실행:** 모든 후보가 처리된 뒤 투자 적격 기업들을 공용 순위 규칙으로 정렬하고 보고서 대상 기업을 선택합니다.

**확인:** `ranking`, `selected_company_id`. 투자 적격이 없으면 선정값이 `None`인 경로도 유효하지만, 앞 단계 오류나 자격 요건 확인불가 때문에 모든 기업이 분석에서 제외됐다면 투자 분석까지 검증된 것으로 해석하지 않습니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("rank")


## 16. 최종 보고서 — report

**실제 실행:** 선정된 기업의 `evaluation_results`를 사용해 보고서를 생성합니다.

**확인:** SUMMARY·본문·REFERENCE, 실제 인용된 근거, RAG ID의 4자리 청크 순번(`TEC-08-0001` 등)이 REFERENCE에 포함되는지 확인합니다. 보고서가 만들어졌다는 사실만으로 앞 단계 오류가 해결되는 것은 아닙니다.

In [ ]:
if EXECUTION_MODE == "step":
    run_next("report")


## 17. 현재까지의 실행 요약

어느 단계에서 멈췄든 실행할 수 있습니다. 노드 실행을 추가하지 않고 현재까지의 상태와 오류만 확인합니다.

In [ ]:
s = current_state()
print(json.dumps({
    "실제 노드 호출 수": len(history),
    "노드별 호출 횟수": dict(Counter(item["노드"] for item in history)),
    "다음 노드": list(next_nodes()),
    "현재 기업": company_summary(s["current_company"]) if s.get("current_company") else None,
    "완료 기업 수": len(s.get("evaluation_results", [])),
    "오류": s.get("errors", []),
    "순위": s.get("ranking", []),
    "선정 기업": s.get("selected_company_id"),
}, ensure_ascii=False, indent=2, default=str))


## 18. 생성된 보고서 보기·저장

보고서가 실제로 생성된 뒤에만 `app.py`와 같은 위치인 `outputs/report.md`에 저장합니다. 실행 중간 상태는 이 셀에서 보고서로 저장하지 않습니다.

In [ ]:
s = current_state()
if next_nodes() or not s.get("final_report"):
    print("아직 최종 보고서 생성이 끝나지 않았습니다. 다음 노드:", next_nodes())
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / "report.md"
    path.write_text(s["final_report"], encoding="utf-8")
    print(f"보고서 저장: {path}")
    display(Markdown(s["final_report"]))


## 19. 실제 상태·실행 순서 기록 저장

실제 최종/중간 상태를 `state.json`, 모든 노드의 순서·기업ID·출력·시간을 `events.json`으로 로컬 `.cache/notebook_pipeline/`에 저장합니다. 커널 재시작 뒤 자동 재개하는 체크포인트는 아니며, 결과와 오류를 다시 살펴보기 위한 기록입니다. API 키나 환경 변수는 저장하지 않습니다.


In [ ]:
if not started:
    print("아직 실행된 노드가 없습니다.")
else:
    folder = ROOT / ".cache" / "notebook_pipeline"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "state.json").write_text(json.dumps(current_state(), ensure_ascii=False, indent=2), encoding="utf-8")
    (folder / "events.json").write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"실제 상태·실행 기록 저장: {folder}")
